# LangChain Fundamentals to Intermediate — Groq Edition

**Topics covered:** LLMs and Chat Models, Prompt Templates, Messages and Roles, Output Parsers, LCEL, Chains, Runnable Interfaces, and Conversation Memory / Chat History.

This notebook is written for beginners. Each concept starts with a plain-English explanation, then a practical example, expected behavior, and a small exercise.

> **Important:** All model calls in this notebook use **Groq** through `ChatGroq`. No OpenAI model is used. You need a Groq API key and an internet connection for model examples.

## Learning map

Think of a LangChain application as a small conversation factory:

1. **Chat model:** the engine that generates an answer.
2. **Prompt template:** the instruction form you fill in.
3. **Messages and roles:** labels that tell the model who is speaking and what kind of message it is.
4. **Output parser:** a formatter that turns the model's response into a useful Python value.
5. **LCEL:** the way you connect steps using `|`.
6. **Chains:** connected steps that perform a task.
7. **Runnable interfaces:** common methods that let you invoke, stream, or batch-run those steps.
8. **Chat history:** the conversation notes supplied again so the model can understand earlier turns.

## 0. Install the required packages

Run this cell once in your notebook environment. If you install packages while Jupyter is already running and imports fail, restart the kernel and run the notebook again.

In [ ]:
%pip install -U langchain langchain-core langchain-groq

## 1. Configure Groq

A model API key is like a password that allows your program to use the model service. Do not paste your key into a notebook that you plan to share or upload publicly.

Create a Groq API key in the Groq Console, then enter it when prompted. `getpass` hides the key while you type.

### Create the Groq chat model

`ChatGroq` is LangChain's chat-model integration for Groq. The model name must be one that is currently available to your Groq account. The example below uses `llama-3.3-70b-versatile`; if Groq changes model availability, replace it with a supported model from your Groq Console.

- `temperature=0`: more consistent, less creative answers.
- A higher temperature: usually more varied answers.
- `model.invoke(...)`: send one request and wait for its response.

In [ ]:
from langchain_groq import ChatGroq

# Keep the model name configurable because provider model availability can change.
MODEL_NAME = "openai/gpt-oss-120b"
GROQ_API_KEY =""

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0
    api_key=GROQ_API_KEY
)
print("Groq chat model initialized.")


Groq chat model initialized.


## 2. LLMs and Chat Models

### What is an LLM?

LLM means **Large Language Model**. Imagine a very advanced text prediction engine trained on a large amount of text. Given your input, it predicts a useful continuation.

Examples of tasks:
- Explain a Python concept.
- Summarize a paragraph.
- Draft an email.
- Generate code.

### What is a Chat Model?

A chat model is designed to work with conversation-style messages. Instead of giving only one plain text string, you can send messages with roles such as system, human, and assistant.

**Simple difference:**
- Traditional text completion: you provide text and ask the model to continue it.
- Chat model: you provide a structured conversation and receive an assistant message.

In LangChain, `ChatGroq` is a chat model. It returns an `AIMessage` object, and the generated text is usually in `.content`.

In [2]:
from langchain_core.messages import HumanMessage

answer = llm.invoke([
    HumanMessage(content="What is a database? Explain with a school example in 5 lines .")
])

print("Python object type:", type(answer))
print("Answer:", answer.content)

Python object type: <class 'langchain_core.messages.ai.AIMessage'>
Answer: A database is an organized collection of data that can be easily accessed, managed, and updated.  
Think of a school’s student information system: each student’s record (name, grade, contact, courses) is a row in a table.  
The table’s columns define the type of information stored (e.g., “Name,” “Grade,” “Phone”).  
Teachers can query the database to find all students in a particular class or to pull a list of honor‑roll candidates.  
Because the data is structured, the school can quickly add new students, modify existing records, and generate reports.


### Practical example: same model, different questions

The model is the engine. You can reuse the same `llm` object for many tasks.

In [3]:
questions = [
    "Explain an API in one simple sentence.",
    "Explain a database table using a classroom example.",
    "What is Python used for? Give three examples."
]

for question in questions:
    result = llm.invoke(question)
    print("\nQUESTION:", question)
    print("ANSWER:", result.content)


QUESTION: Explain an API in one simple sentence.
ANSWER: An API (Application Programming Interface) is a set of rules that lets one software program talk to and use the features of another program.

QUESTION: Explain a database table using a classroom example.
ANSWER: ## What a Database Table Is – A Classroom Analogy  

Think of a **database table** as a neatly organized spreadsheet that lives inside a computer.  
If you’ve ever used a paper‑based grade book or a class roster, you already have a mental picture of what a table looks like. Below we’ll walk through a concrete classroom‑themed example, point out the key parts of a table, and show how tables can work together.

---

## 1. The “Students” Table – Your Class Roster  

| **Column (Field)** | **What It Stores** | **Real‑world Analogy** |
|--------------------|--------------------|------------------------|
| `StudentID` (PK)   | A unique number for each student (e.g., 101, 102…) | The student’s ID badge number – no two students 

### Check your understanding

1. What does LLM stand for?
2. Is `ChatGroq` a prompt template or a chat model?
3. What does `.content` give you?

**Try it:** Ask the model to explain an EC2 instance to a complete beginner.

## 3. Prompt Templates

### Layman's explanation

A prompt is the instruction or question you give to the model.

A **prompt template** is a reusable instruction form with blank spaces. Instead of rewriting the whole prompt every time, you fill in variables.

Think of a printed form:

`Dear {name}, your appointment is on {date}.`

Here, `name` and `date` are placeholders. In LangChain, `{topic}` works the same way.

In [4]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words. Give one real-world example."
)

formatted_prompt = prompt.format(topic="Docker")
print(formatted_prompt)

Explain Docker in simple words. Give one real-world example.


### Send the formatted prompt to Groq

Notice the two separate steps:
1. `prompt.format(...)` fills in the blank.
2. `llm.invoke(...)` sends the completed text to Groq.

In [5]:
question = prompt.format(topic="Kubernetes")
result = llm.invoke(question)
print(result.content)

**Kubernetes in a nutshell**

Think of a **big restaurant kitchen** that has to serve many dishes at the same time, keep the food hot, replace chefs who get sick, and make sure the right ingredients are always stocked.  

- **Containers = individual dishes** – each dish (e.g., a salad, a steak, a dessert) is packaged with everything it needs to be cooked (ingredients, recipe, tools). In tech, a *container* bundles an application and all its libraries so it runs the same everywhere.  

- **Pods = a small serving tray** – sometimes a dish needs two components that must stay together (e.g., a burger and its fries). Kubernetes groups one or more containers that belong together into a *pod*, the smallest unit it manages.  

- **Nodes = kitchen stations** – each station (grill, oven, prep table) can host many trays. In a cluster, a *node* is a physical or virtual machine that runs pods.  

- **Cluster = the whole kitchen** – all the stations together form the *cluster*, the environment where

### ChatPromptTemplate

`PromptTemplate` creates a text prompt. `ChatPromptTemplate` creates a structured set of chat messages with roles.

Use it when you want to give separate instructions to the system and the user.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a patient Python teacher. Explain for a beginner."),
    ("human", "Explain {concept} with a simple example.")
])

messages = chat_prompt.format_messages(concept="Python list")
for message in messages:
    print(type(message).__name__, ":", message.content)

SystemMessage : You are a patient Python teacher. Explain for a beginner.
HumanMessage : Explain Python list with a simple example.


### Practical example: reusable course mentor prompt

In [7]:
mentor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an IT course mentor. Keep the answer practical and beginner-friendly."),
    ("human", "Course: {course}\nStudent level: {level}\nCreate a 3-day starter plan.")
])

mentor_messages = mentor_prompt.format_messages(
    course="AWS",
    level="Complete beginner"
)

mentor_response = llm.invoke(mentor_messages)
print(mentor_response.content)

## 🎓 3‑Day AWS Starter Plan  
**Audience:** Complete beginners – you’ve never used AWS before, but you’re comfortable with a web browser and a bit of computer basics.  
**Goal:** By the end of the three days you will have a working AWS account, know how to move around the console, and have launched three core services (IAM, EC2, S3). You’ll also have a solid foundation for the next steps (VPC, databases, serverless, etc.).  

---

## 📅 Overview

| Day | Theme | Approx. Time | What You’ll Walk Away With |
|-----|-------|--------------|----------------------------|
| **Day 1** | **Account & Identity** – set up, secure, explore the console | 2‑3 h | Free‑tier account, IAM user, MFA, console navigation |
| **Day 2** | **Compute** – launch a virtual server (EC2) and connect to it | 3‑4 h | Running EC2 instance, SSH/RDP basics, simple web server |
| **Day 3** | **Storage & Basics of Networking** – S3 bucket, static website, intro to VPC | 3‑4 h | S3 bucket with public file, static site, VPC 

### Common mistake: missing a variable

If a template contains `{topic}`, you must supply `topic` when formatting or invoking it. A missing value causes an error because LangChain cannot fill the blank.

In [ ]:
# Uncomment to see the missing-variable error:
# prompt.format()

### Exercise

Create a `PromptTemplate` with `{technology}` and `{experience_level}`. Ask Groq to explain the technology using an example suitable for that experience level.

## 4. Messages and Roles

### Layman's explanation

A chat conversation is like a play script. Each line has a speaker label.

| Role | Meaning | Simple example |
|---|---|---|
| `system` | Instructions that guide the assistant | "You are a Python teacher." |
| `human` | Message from the user | "Explain loops." |
| `ai` / `assistant` | Message from the assistant | "A loop repeats work." |
| `tool` | Output returned by a tool | A search or calculator result |

A role tells the model how to interpret a message. It is not just decoration.

### Create messages directly

LangChain provides message classes. `SystemMessage`, `HumanMessage`, and `AIMessage` make the role explicit in Python.

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

conversation = [
    SystemMessage(content="You are a friendly programming teacher."),
    HumanMessage(content="What is a Python function?"),
    AIMessage(content="A function is a reusable block of code."),
    HumanMessage(content="Give me a small example.")
]

for msg in conversation:
    print(type(msg).__name__, "=>", msg.content)

SystemMessage => You are a friendly programming teacher.
HumanMessage => What is a Python function?
AIMessage => A function is a reusable block of code.
HumanMessage => Give me a small example.


### Send a role-based conversation to Groq

In [9]:
role_response = llm.invoke([
    SystemMessage(content="Explain technical topics using everyday examples."),
    HumanMessage(content="Explain the difference between RAM and a hard disk.")
])

print(role_response.content)

**RAM vs. Hard Disk – explained with everyday objects**

| What it is | Everyday analogy | What it does for the computer | Key traits |
|------------|------------------|------------------------------|------------|
| **RAM (Random‑Access Memory)** | **A kitchen counter** – you put the ingredients you’re actively using right on the counter so you can reach them instantly. | Holds the data and programs the CPU is working on *right now* (open apps, the web page you’re scrolling, the game level you’re playing). | • Very fast – like a clean, spacious counter where everything is within arm’s reach.<br>• Temporary (volatile) – when the power is cut, everything on the counter is cleared, just like you’d wipe the counter clean. |
| **Hard Disk (HDD/SSD)** | **A pantry or a filing cabinet** – you store food, dishes, and paperwork there for the long term. You have to open the cabinet and pull out what you need. | Stores everything permanently: OS, apps, photos, videos, documents, etc. When you lau

### Why use roles?

Suppose you write "Answer in simple English" as a system instruction. That instruction guides the assistant. The human message then contains the actual question. Separating them makes prompts easier to maintain and reuse.

**Exercise:** Change the system message to "Answer like an interviewer." Ask the human question: "What is Python?" Compare the response.

## 5. Output Parsers

### Layman's explanation

A model may return a rich message object. An output parser converts that response into a format that is easier for your Python program to use.

Think of a parser as a translator between the model's response and the format your application expects.

Common parsers:
- `StrOutputParser`: extracts plain text.
- `JsonOutputParser`: helps parse structured JSON output.
- `PydanticOutputParser`: parses output into a validated Pydantic model.

A parser does not magically guarantee that the model follows the requested format. Always handle invalid or unexpected output.

### 5.1 StrOutputParser

Without a parser, `llm.invoke(...)` returns an `AIMessage`. With `StrOutputParser`, the chain returns a Python string.

In [10]:
from langchain_core.output_parsers import StrOutputParser

text_parser = StrOutputParser()

message_result = llm.invoke("Give one sentence explaining Python.")
string_result = text_parser.invoke(message_result)

print("Before parser:", type(message_result))
print("After parser:", type(string_result))
print(string_result)

Before parser: <class 'langchain_core.messages.ai.AIMessage'>
After parser: <class 'langchain_core.messages.base.TextAccessor'>
Python is a high‑level, interpreted programming language known for its readable syntax, dynamic typing, and extensive standard library that enables rapid development across many domains.


### 5.2 JsonOutputParser

Ask the model for a small JSON object. `JsonOutputParser` attempts to parse the response into Python data such as a dictionary.

For reliable applications, use structured-output features where supported, validate the result, and handle parsing failures.

In [11]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return only valid JSON. Do not add Markdown fences."),
    ("human", "Give details for this course: {course}. Use keys: course, duration_days, difficulty.")
])

json_chain = json_prompt | llm | JsonOutputParser()

try:
    course_data = json_chain.invoke({"course": "Python"})
    print(type(course_data))
    print(course_data)
except Exception as e:
    print("Could not parse model output:", e)

<class 'dict'>
{'course': 'Python', 'duration_days': 30, 'difficulty': 'Beginner'}


### 5.3 PydanticOutputParser

Pydantic lets you define the shape and types of the data you expect. The parser can validate the model's response against that schema.

In [12]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class CourseInfo(BaseModel):
    course: str = Field(description="Course name")
    duration_days: int = Field(description="Course duration in days")
    difficulty: str = Field(description="Beginner, intermediate, or advanced")

pydantic_parser = PydanticOutputParser(pydantic_object=CourseInfo)

structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer according to these formatting instructions:\n{format_instructions}"),
    ("human", "Create course information for {course}.")
])

structured_chain = (
    structured_prompt
    | llm
    | pydantic_parser
)

try:
    course_info = structured_chain.invoke({
        "course": "Python fundamentals",
        "format_instructions": pydantic_parser.get_format_instructions()
    })
    print(course_info)
    print("Course:", course_info.course)
    print("Duration:", course_info.duration_days)
except Exception as e:
    print("Validation or parsing error:", e)

course='Python Fundamentals' duration_days=30 difficulty='Beginner'
Course: Python Fundamentals
Duration: 30


### Exercise

Ask Groq to return a JSON object containing `technology`, `use_case`, and `beginner_tip`. Parse it and print only the `beginner_tip` value.

## 6. LCEL — LangChain Expression Language

### What is LCEL?

LCEL is a way to connect LangChain components into a workflow. The most recognizable feature is the pipe operator `|`.

Think about a water pipeline:
- Water enters the first pipe.
- The first pipe does its job and passes the result forward.
- The next pipe receives that result.
- The final output comes at the end.

In LCEL, data flows from left to right.

Example:

`prompt | llm | parser`

Meaning:
1. The prompt prepares messages.
2. The model generates a response.
3. The parser extracts plain text.

This connected workflow is called a **chain**.

### Build your first LCEL chain

In [13]:
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a beginner-friendly IT trainer."),
    ("human", "Explain {topic} in simple words and give one example.")
])

simple_chain = simple_prompt | llm | StrOutputParser()

output = simple_chain.invoke({"topic": "Load balancer"})
print(output)

**What a Load Balancer Does – in Plain English**

Imagine you run a popular coffee shop. When only a few customers come in, one barista can serve everyone quickly. But as the shop gets busier, the line grows and customers start waiting a long time. To keep service fast, you hire a second (or third) barista and give each new customer the choice of which barista will take their order. Now the work is split, the line stays short, and everyone gets their coffee faster.

A **load balancer** works the same way, but for computer traffic instead of coffee orders. It sits in front of a group of servers (the “baristas”) and decides which server should handle each incoming request (the “customer”). By spreading the work evenly, it:

* **Prevents any single server from getting overloaded.**
* **Keeps the application fast and responsive.**
* **Provides a safety net – if one server fails, the others keep working.**

---

### Simple Analogy

| Coffee Shop | Load‑Balancing Analogy |
|------------|----

### Understand each stage separately

You can run each component individually to see what it receives and returns.

In [14]:
prompt_value = simple_prompt.invoke({"topic": "Load balancer"})
print("Prompt stage type:", type(prompt_value))
print("Prompt stage value:", prompt_value)

model_message = llm.invoke(prompt_value)
print("\nModel stage type:", type(model_message))
print("Model content:", model_message.content)

plain_text = StrOutputParser().invoke(model_message)
print("\nParser stage type:", type(plain_text))
print("Final output:", plain_text)

Prompt stage type: <class 'langchain_core.prompt_values.ChatPromptValue'>
Prompt stage value: messages=[SystemMessage(content='You are a beginner-friendly IT trainer.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Load balancer in simple words and give one example.', additional_kwargs={}, response_metadata={})]

Model stage type: <class 'langchain_core.messages.ai.AIMessage'>
Model content: **What a Load Balancer Does – in Plain English**

Imagine you run a popular coffee shop. When a lot of customers walk in at the same time, you don’t want everyone to line up at a single counter and wait forever. So you open a second (or third) counter and have a friendly staff member at the entrance who tells each arriving customer which counter to go to. The goal is to keep the lines short, make sure no single counter gets overwhelmed, and give every customer a quick, smooth experience.

A **load balancer** works the same way, but for computer traffic instead of coffee

### Why LCEL is useful

- Makes workflows readable.
- Components can be reused.
- Supports common execution methods such as `invoke`, `stream`, and `batch`.
- Makes it easier to add parallel steps and branching logic.

**Exercise:** Build `prompt | llm | StrOutputParser()` to explain `{cloud_service}`. Invoke it with `"Amazon S3"`.

## 7. Chains

### Layman's explanation

A chain is a sequence of steps that work together to complete a task.

Example: an IT learning-plan chain:
1. Receive a topic and student level.
2. Build a prompt.
3. Ask Groq to generate a plan.
4. Convert the response to plain text.

In LCEL, a chain is often created by joining runnables with `|`. The chain itself can be called with `.invoke()`.

### 7.1 Basic chain

In [15]:
basic_chain = (
    ChatPromptTemplate.from_template(
        "Explain {topic} to a {level} student in 5 bullet points."
    )
    | llm
    | StrOutputParser()
)

print(basic_chain.invoke({
    "topic": "Amazon EC2",
    "level": "beginner"
}))

- **What it is:** Amazon Elastic Compute Cloud (EC2) is a service that lets you rent virtual computers (called “instances”) in Amazon’s data centers, so you can run applications, host websites, or do any kind of computing without buying your own hardware.  

- **Pay‑as‑you‑go:** You only pay for the time the instance is running (by the second or hour), just like paying for electricity. You can start, stop, or terminate an instance whenever you want, and you’re not locked into a long‑term purchase.  

- **Choose the size you need:** EC2 offers many “instance types” (e.g., small, medium, large, GPU‑powered, memory‑optimized). Each type has a specific amount of CPU, RAM, storage, and networking capacity, so you can pick the one that matches your workload and budget.  

- **Scalable on demand:** If your app suddenly gets more traffic, you can launch additional instances automatically (using Auto Scaling) or switch to bigger instance types. When demand drops, you can shut them down to save 

### 7.2 Sequential chain

A sequential chain runs one task after another. The output from one step becomes the input to the next step.

Here, step one creates a short explanation. Step two creates quiz questions based on that explanation.

In [16]:
explanation_chain = (
    ChatPromptTemplate.from_template("Explain {topic} in 4 simple sentences.")
    | llm
    | StrOutputParser()
)

quiz_chain = (
    ChatPromptTemplate.from_template(
        "Based only on this explanation, create 3 beginner quiz questions:\n{explanation}"
    )
    | llm
    | StrOutputParser()
)

explanation = explanation_chain.invoke({"topic": "Python dictionary"})
quiz = quiz_chain.invoke({"explanation": explanation})

print("EXPLANATION:\n", explanation)
print("\nQUIZ:\n", quiz)

EXPLANATION:
 A Python dictionary is a collection of key‑value pairs, where each unique key maps to a value.  
You create one with curly braces, e.g., `my_dict = {"apple": 3, "banana": 5}`.  
You can retrieve, add, or change a value by referencing its key, like `my_dict["apple"] = 4`.  
Dictionaries are unordered (insertion order is preserved in modern Python) and allow fast look‑ups based on the keys.

QUIZ:
 **Beginner Quiz – Python Dictionaries**

---

### Question 1  
Which of the following statements correctly creates a dictionary with the keys `"cat"` and `"dog"` and the values `2` and `5` respectively?

A. `my_animals = ["cat": 2, "dog": 5]`  
B. `my_animals = {"cat": 2, "dog": 5}`  
C. `my_animals = ("cat": 2, "dog": 5)`  
D. `my_animals = {"cat" = 2, "dog" = 5}`  

**Answer:** B  

---

### Question 2  
Given the dictionary  

```python
fruits = {"apple": 3, "banana": 5}
```  

What will be the value of `fruits["apple"]` after executing the line `fruits["apple"] = 4`?

A. `3` 

### 7.3 Combine steps into one sequential LCEL chain

`RunnablePassthrough.assign(...)` can add a new value to the dictionary while keeping the existing values. The second step can then use the first step's output.

In [17]:
from langchain_core.runnables import RunnablePassthrough

sequential_chain = (
    RunnablePassthrough.assign(
        explanation=explanation_chain
    )
    | RunnablePassthrough.assign(
        quiz=quiz_chain
    )
)

result = sequential_chain.invoke({"topic": "Python dictionary"})
print("Topic:", result["topic"])
print("\nExplanation:", result["explanation"])
print("\nQuiz:", result["quiz"])

Topic: Python dictionary

Explanation: A Python dictionary is a collection of key‑value pairs, where each unique key maps to a value.  
You create one with curly braces, e.g., `my_dict = {"apple": 3, "banana": 5}`.  
You can retrieve, add, or change a value by referencing its key, like `my_dict["apple"] = 4`.  
Dictionaries are unordered (insertion order is preserved in modern Python) and allow fast look‑ups based on the keys.

Quiz: **Beginner Quiz – Python Dictionaries**

---

### Question 1  
Which of the following statements correctly creates a dictionary with the keys `"cat"` and `"dog"` and the values `2` and `7` respectively?

A. `my_animals = ["cat": 2, "dog": 7]`  
B. `my_animals = {"cat": 2, "dog": 7}`  
C. `my_animals = ("cat": 2, "dog": 7)`  
D. `my_animals = {"cat", 2, "dog", 7}`  

**Answer:** B  

---

### Question 2  
Given the dictionary  

```python
fruits = {"apple": 3, "banana": 5}
```

What will be the value of `fruits["apple"]` after executing the line `fruits["ap

### Exercise

Create two steps:
1. Generate a beginner explanation of Terraform.
2. Generate three interview questions based on that explanation.

Connect the steps so the second uses the first step's output.

## 8. Runnable Interfaces

### What is a Runnable?

A Runnable is a LangChain component that follows a common execution interface. A prompt, chat model, parser, and LCEL chain can all be used as runnables.

Think of a standard electrical plug: different devices can use the same type of socket. Similarly, different LangChain components share standard ways to run.

Important methods:
- `invoke(input)`: run once and return the final result.
- `stream(input)`: yield output chunks as they are produced.
- `batch(inputs)`: run multiple inputs.
- `ainvoke(input)`: asynchronous version of invoke.
- `astream(input)`: asynchronous streaming.
- `abatch(inputs)`: asynchronous batch execution.

### 8.1 invoke — one input

In [18]:
runnable_chain = (
    ChatPromptTemplate.from_template("Define {term} in one simple paragraph.")
    | llm
    | StrOutputParser()
)

one_result = runnable_chain.invoke({"term": "API"})
print(one_result)

An API (Application Programming Interface) is a set of clearly defined rules, protocols, and tools that allows different software applications or components to communicate and interact with each other, enabling developers to request specific functionality or data from a service without needing to understand its internal code—essentially a contract that specifies how software pieces can “talk” to one another, whether through function calls, web requests, or other interfaces.


### 8.2 batch — multiple inputs

`batch()` accepts a list of inputs. It returns a list of results in corresponding order. Batch execution can be more efficient than manually invoking each input, depending on the model provider and configuration.

In [19]:
batch_inputs = [
    {"term": "Python"},
    {"term": "Docker"},
    {"term": "Kubernetes"}
]

batch_results = runnable_chain.batch(batch_inputs)

for term, result in zip(["Python", "Docker", "Kubernetes"], batch_results):
    print(f"\n{term}:\n{result}")


Python:
Python is a high‑level, interpreted programming language known for its clear, readable syntax and dynamic typing, which makes it easy to write and maintain code quickly. Created by Guido van Rossum in 1991, it supports multiple programming paradigms—including procedural, object‑oriented, and functional styles—and comes with a vast standard library plus an extensive ecosystem of third‑party packages. These features, together with its cross‑platform nature, have made Python a popular choice for web development, data analysis, scientific computing, artificial intelligence, automation, and many other fields.

Docker:
Docker is an open‑source platform that lets developers package an application and all its dependencies—code, runtime, system tools, libraries, and settings—into a lightweight, portable container. These containers run consistently across any environment that supports Docker, from a developer’s laptop to on‑premises servers or cloud infrastructure, eliminating the “it w

### 8.3 stream — receive output chunks

Streaming lets your application display parts of a response as they arrive instead of waiting for the complete answer.

In [20]:
for chunk in runnable_chain.stream({"term": "CI/CD"}):
    print(chunk, end="", flush=True)
print()

Continuous Integration/Continuous Delivery (CI/CD) is a set of automated software development practices that streamline how code changes are built, tested, and released. With CI, developers frequently merge their work into a shared repository where automated builds and tests run to catch errors early. CD extends this by automatically deploying each validated build to staging or production environments, ensuring that new features, fixes, and improvements can be delivered quickly, reliably, and with minimal manual intervention. Together, CI/CD accelerates the development cycle, improves code quality, and reduces the risk of deployment failures.


### 8.4 async methods

Asynchronous methods are useful in applications that handle many requests without blocking the event loop. In Jupyter, top-level `await` is supported.

In [21]:
async_result = await runnable_chain.ainvoke({"term": "Serverless computing"})
print(async_result)

# Example of async batch:
async_results = await runnable_chain.abatch([
    {"term": "Git"},
    {"term": "Linux"}
])
print(async_results)

Serverless computing is a cloud‑based execution model where developers write and deploy code without managing the underlying servers or infrastructure; the cloud provider automatically provisions, scales, and bills only for the exact compute resources used during each function’s execution, allowing teams to focus solely on application logic while the platform handles capacity, availability, and maintenance.
['Git is a distributed version‑control system that tracks changes to files and coordinates work among multiple developers. Every user has a complete copy of the project’s history on their own machine, allowing them to create, merge, and revert changes locally without needing a central server. Git records each change as a snapshot (a commit) identified by a unique hash, supports branching to experiment or develop features in isolation, and provides tools for synchronizing branches across repositories, making collaboration, code review, and rollback straightforward and efficient.', 'L

### 8.5 RunnableLambda

`RunnableLambda` turns an ordinary Python function into a runnable. This is useful when you need custom Python logic inside an LCEL workflow.

In [22]:
from langchain_core.runnables import RunnableLambda

def make_uppercase(text):
    return text.upper()

uppercase_runnable = RunnableLambda(make_uppercase)

print(uppercase_runnable.invoke("hello langchain"))

HELLO LANGCHAIN


### 8.6 RunnableParallel

`RunnableParallel` runs multiple runnable steps using the same input and collects their results in a dictionary.

Example: ask Groq for both a short explanation and an analogy for the same topic.

In [23]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    "simple_explanation": (
        ChatPromptTemplate.from_template("Explain {topic} in two simple sentences.")
        | llm
        | StrOutputParser()
    ),
    "real_world_analogy": (
        ChatPromptTemplate.from_template("Give a real-world analogy for {topic}.")
        | llm
        | StrOutputParser()
    )
})

parallel_result = parallel_chain.invoke({"topic": "Database"})
print("Explanation:", parallel_result["simple_explanation"])
print("\nAnalogy:", parallel_result["real_world_analogy"])

Explanation: A database is an organized collection of data stored electronically so it can be easily accessed, managed, and updated. It lets users retrieve specific information quickly without having to search through raw files or documents.

Analogy: ### The Library Analogy – A Real‑World Way to Picture a Database

| Real‑World Element | Database Counterpart | What It Shows You |
|--------------------|----------------------|-------------------|
| **Library building** | **Database server** | The physical machine (or cloud service) that houses everything. |
| **Bookshelves** | **Tables (or collections)** | Organized containers that hold items of the same type (e.g., all mystery novels, all science textbooks). |
| **Books** | **Rows/records** | Individual data entries. Each book has its own title, author, ISBN, etc., just as a row has columns of values. |
| **Book index cards / catalog** | **Indexes** | A fast‑lookup system that tells you exactly which shelf and position a book lives on,

### 8.7 RunnablePassthrough

`RunnablePassthrough` passes input through without changing it. It is often useful when you want to preserve original input while adding generated fields.

In [24]:
from langchain_core.runnables import RunnablePassthrough

passthrough_example = RunnablePassthrough()
sample = {"name": "Veera", "course": "LangChain"}

print(passthrough_example.invoke(sample))

{'name': 'Veera', 'course': 'LangChain'}


### Exercise

Use `RunnableParallel` to generate:
- a beginner definition of AWS Lambda;
- two use cases for AWS Lambda;
- one interview question about AWS Lambda.

## 9. Conversation Memory / Chat History

### Layman's explanation

Imagine talking to a teacher. You ask, "What is Python?" The teacher answers. Then you ask, "Give me an example." A human teacher understands that "it" means Python because they remember the previous question.

A model request is generally independent: the model does not automatically remember earlier calls. Your application must send relevant previous messages again.

**Chat history** is the list of earlier user and assistant messages. It provides context for the next request.

Modern LangChain applications commonly use message history with `RunnableWithMessageHistory`. Older classes such as `ConversationBufferMemory` appear in older tutorials; this notebook demonstrates the runnable-based chat-history pattern.

### 9.1 Chat history without automatic storage

First, see the basic idea: keep messages in a Python list and send the list with the next question.

In [25]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

chat_history = [
    SystemMessage(content="You are a helpful Python teacher."),
    HumanMessage(content="My name is Veera and I am learning Python."),
    AIMessage(content="Nice to meet you, Veera. I can help you learn Python.")
]

next_question = HumanMessage(content="What should I learn first?")
history_response = llm.invoke(chat_history + [next_question])

print(history_response.content)

## Getting Started with Python – A Friendly Road‑Map for Beginners  

Hi Veera! 🎉  
Welcome to the world of Python. It’s a great language for beginners because its syntax is clean, the community is huge, and it’s useful for everything from web development to data science. Below is a step‑by‑step learning path that will give you a solid foundation and keep you motivated.

---

### 1️⃣ Set Up Your Environment  
| What to Do | Why It Matters |
|------------|----------------|
| **Install Python** (download from [python.org](https://www.python.org/downloads/) or use a package manager like `brew`/`apt`). | Gives you the interpreter to run code. |
| **Choose an IDE / Editor** – e.g., VS Code, PyCharm Community, or even the built‑in IDLE. | A good editor provides syntax highlighting, auto‑completion, and debugging tools. |
| **Learn to run scripts** – `python myscript.py` in a terminal, or use the “Run” button in your IDE. | You’ll be able to test code quickly. |
| **Optional:** Install **Anac

### 9.2 RunnableWithMessageHistory

This helper connects a chain to a chat-history store. It:
1. Receives a session ID.
2. Loads the messages stored for that session.
3. Adds the current user message.
4. Runs the chain with the conversation.
5. Stores the new user and assistant messages.

A session ID is like a conversation folder name. Different IDs keep different conversations separate.

In [26]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Demo-only in-memory store. Data is lost when the Python process ends.
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant. Use the conversation history when relevant."),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])

history_chain = history_prompt | llm | StrOutputParser()

chat_with_history = RunnableWithMessageHistory(
    history_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

c:\Users\vvard\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Ask the first question

In [27]:
config = {"configurable": {"session_id": "student-001"}}

response1 = chat_with_history.invoke(
    {"question": "My name is Veera. I am learning Python."},
    config=config
)
print(response1)

C:\Users\vvard\AppData\Local\Temp\ipykernel_11008\2164368254.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Hi Veera! 👋 Great to meet you, and welcome to the world of Python. How’s your learning journey going so far? Are you working on any particular project or topic (like loops, functions, data structures, or something else) that you’d like some help with? Feel free to ask any questions—big or small—I’m here to help!


### Ask a follow-up question

The second call uses the same session ID. The chain can now use the messages stored from the first call.

In [28]:
response2 = chat_with_history.invoke(
    {"question": "What language am I learning?"},
    config=config
)
print(response2)

You’re learning **Python**! 🚀 If you have any questions about it—whether it’s basics, data structures, libraries, or anything else—just let me know. Happy coding, Veera!


### Inspect stored history

The in-memory store is a Python dictionary. Each session ID points to its own message history.

In [29]:
for message in get_session_history("student-001").messages:
    print(type(message).__name__, ":", message.content)

HumanMessage : My name is Veera. I am learning Python.
AIMessage : Hi Veera! 👋 Great to meet you, and welcome to the world of Python. How’s your learning journey going so far? Are you working on any particular project or topic (like loops, functions, data structures, or something else) that you’d like some help with? Feel free to ask any questions—big or small—I’m here to help!
HumanMessage : What language am I learning?
AIMessage : You’re learning **Python**! 🚀 If you have any questions about it—whether it’s basics, data structures, libraries, or anything else—just let me know. Happy coding, Veera!


### Different session means separate conversation

In [30]:
other_config = {"configurable": {"session_id": "student-002"}}

response3 = chat_with_history.invoke(
    {"question": "Do you know my name?"},
    config=other_config
)
print(response3)

I don’t know your name yet—unless you’ve told me before in this conversation, I don’t have that information. If you’d like, feel free to share it and I’ll be happy to use it when we chat!


### Important memory notes

- `InMemoryChatMessageHistory` is for demonstrations and simple temporary sessions. It is not durable storage.
- Restarting the kernel clears this in-memory history.
- A production app usually stores history in a persistent database or a supported history backend.
- Keep each user's session isolated. Do not accidentally share one session ID between different users.
- Sending more history uses more tokens. Real applications often limit, summarize, or retrieve relevant history.
- Chat history is not the same as a model permanently learning or changing its weights. It is context supplied with requests.

### Exercise

1. Start a new session with ID `"student-003"`.
2. Tell the assistant your favorite programming language.
3. Ask "What is my favorite language?" using the same session ID.
4. Repeat the question with a different session ID. Compare the results.

## 10. Mini Project — IT Course Mentor

This project combines the topics in one workflow:
- Chat model: Groq
- Prompt template: reusable mentor instructions
- Roles: system and human
- Output parser: plain text
- LCEL: pipe operator
- Chain: connected components
- Runnable interface: invoke
- Chat history: same session remembers earlier turns

In [31]:
project_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced IT course mentor. Be practical, encouraging, and clear."),
    ("placeholder", "{chat_history}"),
    ("human", "Student level: {level}\nQuestion: {question}")
])

project_chain = project_prompt | llm | StrOutputParser()

course_mentor = RunnableWithMessageHistory(
    project_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

project_config = {"configurable": {"session_id": "mentor-demo"}}

answer = course_mentor.invoke(
    {
        "level": "Beginner",
        "question": "Create a 5-day learning plan for Python fundamentals."
    },
    config=project_config
)

print(answer)

c:\Users\vvard\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 5‑Day Python Fundamentals Learning Plan  
**Audience:** Complete beginners (no prior programming experience)  
**Goal:** By the end of the week you’ll be comfortable writing, running, and debugging simple Python scripts, understand core concepts (variables, data types, control flow, functions, and basic I/O), and have a small project you can show off.

---

### General Tips for All Days
| Tip | Why it helps |
|-----|--------------|
| **Set a 60‑90 min “focus block”** each day (no phone, no email). | Consistency beats cramming. |
| **Write code, don’t just read.** Type every example yourself. | Muscle memory = confidence. |
| **Use the built‑in REPL** (`python` or `python3` in a terminal) for quick experiments. | Immediate feedback reinforces concepts. |
| **Save every script** in a folder called `python‑starter/`. Name files descriptively (`01_variables.py`, `03_loops.py`, …). | You’ll have a personal reference library to revisit later. |
| **Ask “What if…?”** After each new concept

### Follow-up in the same project session

In [32]:
follow_up = course_mentor.invoke(
    {
        "level": "Beginner",
        "question": "Based on the plan you gave me, what should I practise on Day 1?"
    },
    config=project_config
)

print(follow_up)

### Day 1 – What to **practice** (hands‑on, not just read)

Below is a bite‑size “practice checklist” you can work through in a single 60‑90 minute focus block.  
Treat each bullet as a **mini‑exercise** – type the code, run it, then tweak it a little before moving on.  

---

#### 1️⃣ Set‑up & “Hello, World!”  
| Step | Action |
|------|--------|
| **a.** Open a terminal (or PowerShell / Command Prompt) and type `python --version` to confirm the interpreter is installed. | You should see something like `Python 3.12.x`. |
| **b.** Open VS Code (or any editor) and create a folder called `python‑starter`. Inside it, create a file named `01_hello.py`. | This will be your first script file. |
| **c.** Write the single line: <br>`print("Hello, World!")` | Save and run it with `python 01_hello.py`. |
| **d.** **Experiment** – change the text, add another `print` line, or use single quotes instead of double quotes. | Observe the output each time. |

---

#### 2️⃣ Variables & Basic Data Types 

## 11. Common errors and troubleshooting

| Problem | Possible reason | What to check |
|---|---|---|
| Authentication error | Missing or invalid Groq API key | Check `GROQ_API_KEY` |
| Model not found / unavailable | Model ID is not available to your account | Check current model IDs in Groq Console |
| Import error | Package missing or old environment | Run the install cell; restart kernel |
| Missing prompt variable | Template placeholder was not supplied | Match `{variable}` with input dictionary key |
| JSON parsing error | Model returned invalid JSON or extra text | Tighten instructions; validate and handle errors |
| History not recalled | New or different session ID | Reuse the same session ID |
| Rate limit / request error | Provider limit or temporary service issue | Retry responsibly and check Groq status/limits |

Never print or share your API key. If it is exposed, revoke it and create a new one.

## 12. Final revision — concepts in one table

| Concept | Simple meaning | Example in this notebook |
|---|---|---|
| LLM | Language-generation engine | Groq-hosted Llama model |
| Chat model | Model that accepts conversation messages | `ChatGroq` |
| Prompt template | Reusable instruction with blanks | `ChatPromptTemplate` |
| Message role | Who is speaking | system / human / AI |
| Output parser | Converts model output to a useful format | `StrOutputParser` |
| LCEL | Connect components into a workflow | `prompt \| llm \| parser` |
| Chain | A connected sequence of tasks | `simple_chain` |
| Runnable | Component with standard execution methods | `invoke`, `batch`, `stream` |
| Chat history | Earlier messages sent as context | `RunnableWithMessageHistory` |

## Suggested learning order

1. Run sections 1–5 and understand each component separately.
2. Practise the first LCEL chain in section 6.
3. Study chains and runnable methods in sections 7–8.
4. Practise chat history in section 9.
5. Complete the mini project in section 10 without copying the code.

**Remember:** the most important LCEL pattern to understand is:

`Prompt Template → Chat Model → Output Parser`

In code:

`prompt | llm | StrOutputParser()`

That pattern is the foundation for many LangChain applications.

# 13. More Practical Examples — Practice Lab

This section adds more hands-on examples for every major topic. Run these after completing the earlier sections. All model-powered examples use the same Groq `llm` object created near the beginning of this notebook.

## Example 1 — LLM and Chat Model: classify a student question

**Real-world situation:** A training institute receives many student questions. We want the model to identify whether each question is about Python, AWS, DevOps, or General.

This demonstrates that the same chat model can perform a different task simply by changing the instruction.

In [33]:
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the question into exactly one category: Python, AWS, DevOps, or General. Return only the category."),
    ("human", "{student_question}")
])

classification_chain = classification_prompt | llm | StrOutputParser()

student_questions = [
    "What is a Python dictionary?",
    "How does an EC2 instance work?",
    "What is a Jenkins pipeline?",
    "How can I improve my interview communication?"
]

for q in student_questions:
    category = classification_chain.invoke({"student_question": q})
    print(f"Question: {q}\nCategory: {category}\n")

Question: What is a Python dictionary?
Category: Python

Question: How does an EC2 instance work?
Category: AWS

Question: What is a Jenkins pipeline?
Category: DevOps

Question: How can I improve my interview communication?
Category: General



## Example 2 — Prompt Templates: change the output using variables

**Real-world situation:** The same prompt can explain different technologies to students at different levels. Only the input values change.

In [34]:
technology_prompt = ChatPromptTemplate.from_template(
    "Act as an IT trainer. Explain {technology} to a {student_level} student. "
    "Include one analogy and one practical use case. Keep it under 150 words."
)

technology_chain = technology_prompt | llm | StrOutputParser()

print(technology_chain.invoke({
    "technology": "Docker",
    "student_level": "complete beginner"
}))

**Docker in a nutshell (≈130 words)**  

Think of Docker like a **portable lunchbox**. You pack a sandwich, a drink, utensils, and a napkin—all the exact items you need for a specific meal—into one container. No matter which kitchen you go to, you just open the lunchbox and everything works the same way, without having to hunt for ingredients or tools.  

Docker does the same for software. It bundles an application, its code, libraries, and system settings into a **Docker container**. That container runs identically on any computer that has Docker installed, eliminating “it works on my machine” problems.

**Practical use case:** A developer builds a web app (e.g., a Node.js API). By placing the app in a Docker container, the team can ship the container to testers, staging servers, and production—each environment runs the exact same version without extra setup, speeding up deployment and reducing bugs.


### Example 2B — Reuse one template with different inputs

In [35]:
for technology in ["Git", "Terraform", "Kubernetes"]:
    print("\n" + "=" * 50)
    print("TOPIC:", technology)
    print(technology_chain.invoke({
        "technology": technology,
        "student_level": "beginner"
    }))


TOPIC: Git
**Git in a nutshell (≈130 words)**  

Git is a *version‑control system*: it records every change you make to your files, so you can revisit, compare, or restore any past state.

**Analogy:** Think of Git like a **time‑traveling notebook**. Each time you finish a page (a “commit”), Git saves a snapshot of the whole notebook. You can flip back to any page, copy it, or merge notes from different notebooks without losing anything.

**Key ideas**  
- **Repository (repo):** the notebook folder Git tracks.  
- **Commit:** a saved snapshot with a message describing the change.  
- **Branch:** a parallel line of development—like a separate draft of the notebook.  
- **Merge:** combining branches back together.

**Practical use case:** A team building a website can each work on separate features in their own branches. When a feature is ready, they merge it into the main branch, and Git automatically resolves or flags any conflicts, ensuring the site’s code stays consistent.

TOPIC: T

## Example 3 — Messages and Roles: system instruction changes the answer

The human question stays the same, but the system role changes the assistant's behavior. Compare a teacher-style answer with an interviewer-style answer.

In [36]:
question = "What is an API?"

teacher_style = llm.invoke([
    SystemMessage(content="You are a patient teacher. Explain using a daily-life analogy."),
    HumanMessage(content=question)
])

interviewer_style = llm.invoke([
    SystemMessage(content="You are a technical interviewer. Give a concise interview-style answer."),
    HumanMessage(content=question)
])

print("TEACHER STYLE:\n", teacher_style.content)
print("\nINTERVIEW STYLE:\n", interviewer_style.content)

TEACHER STYLE:
 ### Think of an API Like a Restaurant Menu (and the Waiter)

Imagine you’re at a restaurant. You sit down, look at the menu, and decide what you’d like to eat. You don’t go into the kitchen, you don’t have to know how the chef prepares the dish, and you certainly don’t need to know the exact temperature of the oven or the exact timing of each step. All you need to do is:

1. **Read the menu** – see what dishes are available and what you can order.
2. **Tell the waiter your choice** – “I’d like the spaghetti carbonara, please.”
3. **Wait for the food** – the kitchen prepares it, the waiter brings it to you, and you enjoy the meal.

In this analogy:

| Restaurant Element | API Equivalent |
|--------------------|----------------|
| **Menu**           | **API documentation** – a list of what the service can do, the “endpoints” (dishes) you can request, and the format of the request/response (ingredients, portion size, etc.). |
| **Waiter**         | **API call** – the messe

### Example 3B — Keep a conversation as messages

In [37]:
role_based_chat = [
    SystemMessage(content="You are a Python mentor. Keep answers beginner-friendly."),
    HumanMessage(content="What is a Python list?"),
]
first_answer = llm.invoke(role_based_chat)
role_based_chat.append(first_answer)
role_based_chat.append(HumanMessage(content="Can you show a small code example?"))

second_answer = llm.invoke(role_based_chat)
print(second_answer.content)

# Inspect the conversation roles and contents
for item in role_based_chat + [second_answer]:
    print(type(item).__name__, ":", item.content[:180])

Sure! Let’s write a tiny program that lets a user build a **shopping list** and then prints it out nicely.

```python
# shopping_list.py
# A very small program that demonstrates how to create, modify,
# and read a Python list.

def main():
    # 1️⃣ Start with an empty list
    shopping = []

    # 2️⃣ Keep asking the user for items until they type "done"
    while True:
        item = input("Enter an item (or type 'done' to finish): ").strip()
        if item.lower() == "done":
            break          # exit the loop
        if item:           # ignore empty input
            shopping.append(item)   # add the new item to the list

    # 3️⃣ Show the final list
    print("\n🛒 Your shopping list:")
    if not shopping:               # the list is empty
        print("  (nothing yet)")
    else:
        # enumerate gives us a nice numbered list: 1., 2., 3., …
        for i, product in enumerate(shopping, start=1):
            print(f"  {i}. {product}")

if __name__ == "__main__":
    

## Example 4 — Output Parsers: convert an answer into a Python list

`CommaSeparatedListOutputParser` turns comma-separated model output into a Python list. The parser is useful when the next part of your Python program needs individual items.

In [38]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return only a comma-separated list. Do not number the items."),
    ("human", "Give 5 important topics to learn in {subject}.")
])

list_chain = list_prompt | llm | list_parser

topics = list_chain.invoke({"subject": "Python fundamentals"})
print("Python type:", type(topics))
print("Parsed list:", topics)

for index, topic in enumerate(topics, start=1):
    print(index, topic.strip())

Python type: <class 'list'>
Parsed list: ['Variables and Data Types', 'Control Flow (if/else', 'loops)', 'Functions', 'Data Structures (lists', 'tuples', 'dictionaries', 'sets)', 'Modules and Packages']
1 Variables and Data Types
2 Control Flow (if/else
3 loops)
4 Functions
5 Data Structures (lists
6 tuples
7 dictionaries
8 sets)
9 Modules and Packages


### Example 4B — Use parsed data in normal Python code

In [39]:
# This is ordinary Python after parsing.
# You can count, loop over, or filter the returned list.
print("Number of topics:", len(topics))
print("First topic:", topics[0].strip() if topics else "No topics returned")

Number of topics: 9
First topic: Variables and Data Types


## Example 5 — LCEL: build a reusable email-drafting chain

The pipe operator passes the output of each stage to the next stage:

`prompt | llm | parser`

Here, the prompt receives values, Groq drafts the email, and the parser returns plain text.

In [40]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write clear, polite, professional workplace emails."),
    ("human", "Write an email to {recipient} about {purpose}. Keep the tone {tone}.")
])

email_chain = email_prompt | llm | StrOutputParser()

draft = email_chain.invoke({
    "recipient": "the training team",
    "purpose": "tomorrow's Python doubt-clearing session at 10 AM",
    "tone": "friendly and professional"
})

print(draft)

**Subject:** Reminder: Python Doubt‑Clearing Session – Tomorrow at 10:00 AM  

Hi Training Team,

I hope you’re all doing well!

Just a quick reminder that our **Python doubt‑clearing session** will take place **tomorrow (Wednesday, 14 Nov) at 10:00 AM** in the main conference room (or via the Teams link below, for remote participants).

**What to expect**
- A brief recap of the key concepts covered in the recent Python modules.  
- Open floor for any questions, code‑review requests, or troubleshooting challenges you’ve encountered.  
- Tips and best practices for writing clean, efficient Python code.

**How to prepare**
- Bring any scripts, notebooks, or error messages you’d like to discuss.  
- If you’re joining remotely, please use the Teams link: [Insert Teams Meeting Link] and test your audio/video beforehand.  

Your participation will help ensure everyone feels confident moving forward with the curriculum. If you have any specific topics you’d like us to address, feel free to re

### Example 5B — Inspect the output of each LCEL stage

In [41]:
sample_input = {"topic": "AWS IAM"}

# Stage 1: prompt template
stage1 = simple_prompt.invoke(sample_input)
print("Stage 1 output:", type(stage1))
print(stage1)

# Stage 2: chat model
stage2 = llm.invoke(stage1)
print("\nStage 2 output:", type(stage2))
print(stage2.content)

# Stage 3: parser
stage3 = StrOutputParser().invoke(stage2)
print("\nStage 3 output:", type(stage3))
print(stage3)

Stage 1 output: <class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='You are a beginner-friendly IT trainer.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain AWS IAM in simple words and give one example.', additional_kwargs={}, response_metadata={})]

Stage 2 output: <class 'langchain_core.messages.ai.AIMessage'>
**What is AWS IAM?**  

Think of AWS (Amazon Web Services) as a big office building that holds many rooms (services) – like a storage room (S3), a compute lab (EC2), a database vault (RDS), and so on.  

**IAM (Identity & Access Management)** is the security desk at the front door of that building. It decides:

| IAM component | Everyday‑life analogy | What it does in AWS |
|---------------|----------------------|----------------------|
| **User**      | A person who works in the office (e.g., Alice) | An individual identity that can sign in to the AWS console or call APIs |
| **Group**     | A team or department (

## Example 6 — Chains: generate an explanation and then interview questions

This is a two-step workflow. The first chain explains a topic. The second chain receives that explanation and creates questions from it.

In [42]:
topic_explanation = (
    ChatPromptTemplate.from_template("Explain {topic} in 5 beginner-friendly bullet points.")
    | llm
    | StrOutputParser()
)

question_generator = (
    ChatPromptTemplate.from_template(
        "Create 5 interview questions based on this explanation.\n\n{notes}"
    )
    | llm
    | StrOutputParser()
)

notes = topic_explanation.invoke({"topic": "AWS S3"})
interview_questions = question_generator.invoke({"notes": notes})

print("NOTES:\n", notes)
print("\nINTERVIEW QUESTIONS:\n", interview_questions)

NOTES:
 - **What it is:** Amazon S3 (Simple Storage Service) is a cloud‑based place to store any type of file (photos, videos, backups, logs, etc.) that you can access over the internet.  
- **How it works:** Files are saved as **objects** inside **buckets** (think of a bucket as a folder). Each object gets a unique key (its name) and can be retrieved with a simple URL or API call.  
- **Scalable & durable:** S3 automatically expands to hold as much data as you need and replicates it across multiple data centers, giving 99.999999999% (11 9’s) durability—your data stays safe even if hardware fails.  
- **Pay‑as‑you‑go:** You only pay for the storage you actually use (GB‑months), the data you transfer out of S3, and any extra features (like versioning or lifecycle rules). No upfront servers or capacity planning required.  
- **Common uses for beginners:**  
  1. **Static website hosting** – serve HTML, CSS, images directly from a bucket.  
  2. **Backup & archive** – keep copies of impor

## Example 7 — RunnableLambda: add Python logic into a chain

A runnable does not have to call an LLM. It can also run your own Python function.

In this example, a Python function prepares a topic label before the prompt is sent to Groq.

In [43]:
from langchain_core.runnables import RunnableLambda

def prepare_topic(data):
    topic = data["topic"].strip().title()
    return {"topic": topic, "level": data.get("level", "beginner")}

prepare_input = RunnableLambda(prepare_topic)

prepared = prepare_input.invoke({
    "topic": "  amazon ec2 ",
    "level": "beginner"
})
print(prepared)

prepared_chain = (
    prepare_input
    | ChatPromptTemplate.from_template("Explain {topic} for a {level} student.")
    | llm
    | StrOutputParser()
)

print(prepared_chain.invoke({"topic": "  amazon ec2 ", "level": "beginner"}))

{'topic': 'Amazon Ec2', 'level': 'beginner'}
## Amazon EC2 – “Computers in the Cloud” explained for a beginner

### 1. What is EC2?
- **EC2 = Elastic Compute Cloud** – a service that lets you run **virtual computers** (called *instances*) on Amazon’s huge network of data‑centers.
- Think of it like renting a fully‑furnished apartment (the instance) instead of buying a house (a physical server). You get the furniture (CPU, memory, storage, network) already set up, and you only pay for the time you stay.

### 2. Why would you use it?
| Need | How EC2 helps |
|------|----------------|
| **Run a website** | Spin up a web‑server instance, point your domain to it, and it’s live. |
| **Test software** | Launch a throw‑away instance, try out code, then delete it – no hardware to buy or maintain. |
| **Scale when traffic spikes** | Add more instances automatically (auto‑scaling) so your app never slows down. |
| **Run big data jobs** | Choose powerful instance types with lots of CPU, RAM, or GP

## Example 8 — RunnableParallel: perform two analyses from the same input

Both branches receive the same topic. One generates advantages; the other generates disadvantages. Their outputs are collected in a dictionary.

In [44]:
pros_chain = (
    ChatPromptTemplate.from_template("Give 3 advantages of {technology}.")
    | llm
    | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("Give 3 limitations of {technology}.")
    | llm
    | StrOutputParser()
)

comparison_chain = RunnableParallel({
    "advantages": pros_chain,
    "limitations": cons_chain
})

comparison = comparison_chain.invoke({"technology": "Docker"})

print("ADVANTAGES:\n", comparison["advantages"])
print("\nLIMITATIONS:\n", comparison["limitations"])

ADVANTAGES:
 Here are three key advantages of using Docker:

| # | Advantage | Why it matters |
|---|-----------|----------------|
| 1 | **Consistent, portable environments** | A Docker image bundles the application code **and** all its dependencies (libraries, runtime, OS‑level settings). This means the same container runs the same way on a developer’s laptop, a test server, or a production cluster, eliminating the “it works on my machine” problem. |
| 2 | **Fast, lightweight deployment** | Containers share the host kernel and start in seconds, unlike full virtual machines that need to boot an entire OS. This speed enables rapid scaling, continuous integration/continuous deployment (CI/CD) pipelines, and efficient use of hardware resources. |
| 3 | **Isolation and reproducibility** | Each container runs in its own isolated namespace (process, network, filesystem). This isolation prevents conflicts between services, improves security, and makes it easy to roll back or recreate an exact

## Example 9 — RunnableBranch: choose a path based on input

A branch is like a road junction. A condition decides which path to follow.

This example uses a simple Python condition. If the input says `beginner`, the beginner prompt is selected; otherwise, the advanced prompt is selected.

In [45]:
from langchain_core.runnables import RunnableBranch

beginner_path = (
    ChatPromptTemplate.from_template("Explain {topic} using very simple words and an analogy.")
    | llm
    | StrOutputParser()
)

advanced_path = (
    ChatPromptTemplate.from_template("Explain {topic} with technical details and architecture considerations.")
    | llm
    | StrOutputParser()
)

branch_chain = RunnableBranch(
    (lambda x: x["level"].lower() == "beginner", beginner_path),
    advanced_path
)

print(branch_chain.invoke({"topic": "Kubernetes", "level": "beginner"}))

**Kubernetes in a nutshell**

Think of a **big restaurant** that wants to serve lots of customers quickly, keep the food fresh, and make sure the kitchen never stops working even if something goes wrong.  
Kubernetes is the “restaurant manager” that makes all of that happen for computer programs (called *containers*).

---

## The restaurant analogy

| Kubernetes piece | Restaurant equivalent | What it does (in plain words) |
|------------------|-----------------------|------------------------------|
| **Container**    | A single dish (e.g., a bowl of soup) | The smallest unit of work – a piece of software that does one job. |
| **Pod**          | A tray that holds one or a few dishes that belong together (e.g., soup + bread) | The smallest group Kubernetes runs. All dishes on the same tray share the same “table” (network, storage). |
| **Node**         | A kitchen station (stove, oven, prep area) | A physical or virtual machine that actually runs the trays (pods). |
| **Cluster**     

## Example 10 — RunnablePassthrough.assign: preserve input and add a result

Suppose the original input contains a topic. We want to keep that topic and add a generated explanation. `assign()` adds the new field while preserving the original dictionary fields.

In [46]:
explain_only = (
    ChatPromptTemplate.from_template("Explain {topic} in 3 simple sentences.")
    | llm
    | StrOutputParser()
)

enriched_chain = RunnablePassthrough.assign(explanation=explain_only)

enriched_result = enriched_chain.invoke({"topic": "Terraform"})
print("Original topic:", enriched_result["topic"])
print("\nAdded explanation:", enriched_result["explanation"])

Original topic: Terraform

Added explanation: Terraform is an open‑source tool that lets you describe your cloud (or on‑premises) infrastructure as code using a simple, declarative language.  
When you run `terraform apply`, it reads that code, figures out the differences between the desired state and what actually exists, and then creates, updates, or deletes resources to make them match.  
Because the configuration files are version‑controlled, you can track changes, collaborate with teammates, and reliably reproduce the same environment anywhere.


## Example 11 — Runnable methods: invoke, batch, and stream

Use the same chain with different execution methods:
- `invoke`: one input, one result.
- `batch`: several inputs, several results.
- `stream`: receive chunks as the response is generated.

In [47]:
method_chain = (
    ChatPromptTemplate.from_template("Explain {topic} in two sentences.")
    | llm
    | StrOutputParser()
)

# invoke
print("INVOKE:")
print(method_chain.invoke({"topic": "Linux"}))

# batch
print("\nBATCH:")
batch_output = method_chain.batch([
    {"topic": "Git"},
    {"topic": "Jenkins"}
])
for item in batch_output:
    print("-", item)

# stream
print("\nSTREAM:")
for chunk in method_chain.stream({"topic": "CI/CD"}):
    print(chunk, end="", flush=True)
print()

INVOKE:
Linux is an open‑source, Unix‑like operating system kernel that serves as the core of countless distributions, providing hardware abstraction, process management, and a rich set of system services. Its collaborative development model and modular design allow anyone to customize, extend, and freely distribute the entire OS for servers, desktops, embedded devices, and more.

BATCH:
- Git is a distributed version‑control system that tracks changes to files by storing snapshots of the entire project history in a lightweight, cryptographically‑hashed database, allowing multiple users to work on the same codebase independently and later merge their contributions. It enables branching, committing, and reverting changes locally without a central server, while still supporting collaboration through remote repositories that can be synchronized via push and pull operations.
- Jenkins is an open‑source automation server that orchestrates the building, testing, and deployment of software pr

## Example 12 — Chat history: remember earlier messages in a session

The model does not automatically remember a previous independent API call. Your application must send the earlier messages again or use a history helper.

This example uses `RunnableWithMessageHistory` and an in-memory store. It is suitable for learning, but the history disappears when the Python process/kernel is restarted.

In [48]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

history_store = {}

def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful learning assistant. Use earlier messages when relevant."),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])

memory_chain = memory_prompt | llm | StrOutputParser()

conversation_chain = RunnableWithMessageHistory(
    memory_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

session_config = {"configurable": {"session_id": "demo-session-01"}}

first = conversation_chain.invoke(
    {"question": "My name is Veera and I am learning AWS."},
    config=session_config
)
print("FIRST RESPONSE:\n", first)

second = conversation_chain.invoke(
    {"question": "What technology am I learning?"},
    config=session_config
)
print("\nFOLLOW-UP RESPONSE:\n", second)

c:\Users\vvard\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


FIRST RESPONSE:
 Hi Veera! 👋 Great to meet you, and welcome to the world of AWS. How can I help you on your learning journey?  

- Are you looking for a study plan or roadmap?  
- Do you have specific services you want to dive into (e.g., EC2, S3, Lambda, IAM, etc.)?  
- Need help with hands‑on labs, exam prep, or troubleshooting a particular issue?  

Let me know what you’re focusing on, and I’ll tailor resources, explanations, and tips just for you!

FOLLOW-UP RESPONSE:
 You’re learning **Amazon Web Services (AWS)**—the cloud‑computing platform that provides on‑demand compute power, storage, databases, networking, machine‑learning tools, and many other services you can use to build and run applications without managing physical servers. 🚀


### Example 12B — Session IDs keep conversations separate

In [49]:
new_session_config = {"configurable": {"session_id": "demo-session-02"}}

separate_answer = conversation_chain.invoke(
    {"question": "Do you know my name?"},
    config=new_session_config
)
print(separate_answer)

print("\nMessages in session 1:", len(get_history("demo-session-01").messages))
print("Messages in session 2:", len(get_history("demo-session-02").messages))

I don’t have any information about your name—unless you’ve shared it with me in this conversation, I wouldn’t know. If you’d like me to address you by a particular name, just let me know!

Messages in session 1: 4
Messages in session 2: 2


## 13. Integrated mini-project — IT course assistant

This combines prompt templates, roles, Groq, LCEL, output parsing, and chat history. Enter a course and a question; the assistant answers using the same session's earlier messages.

In [50]:
assistant_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced IT trainer. Explain concepts clearly to beginners and use practical examples."),
    ("placeholder", "{chat_history}"),
    ("human", "Course: {course}\nStudent question: {question}")
])

assistant_chain = assistant_prompt | llm | StrOutputParser()

it_assistant = RunnableWithMessageHistory(
    assistant_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

assistant_config = {"configurable": {"session_id": "it-course-demo"}}

reply = it_assistant.invoke(
    {
        "course": "Python",
        "question": "Explain the difference between a list and a tuple."
    },
    config=assistant_config
)
print(reply)

c:\Users\vvard\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


**Python Lists vs. Tuples – A Beginner‑Friendly Comparison**

---

## 1. What are they?

| Feature | **List** | **Tuple** |
|---------|----------|-----------|
| **Syntax** | `my_list = [1, 2, 3]` | `my_tuple = (1, 2, 3)` |
| **Mutability** | **Mutable** – you can change, add, or remove items after creation. | **Immutable** – once created, the contents cannot be changed. |
| **Typical Use** | Collections that will be modified (e.g., a shopping cart, a queue of tasks). | Fixed collections that should stay constant (e.g., coordinates, days of the week). |
| **Performance** | Slightly slower because Python has to support all the “change” operations. | Faster look‑ups and smaller memory footprint because the object is read‑only. |
| **Available Methods** | Many (e.g., `append`, `extend`, `pop`, `remove`, `sort`, `reverse`). | Very few (mostly just `count` and `index`). |

---

## 2. Why does mutability matter?

### List – you can **edit** it

```python
fruits = ['apple', 'banana', 'cherry']

### Follow-up question — same session

In [51]:
reply2 = it_assistant.invoke(
    {
        "course": "Python",
        "question": "Show one example of each."
    },
    config=assistant_config
)
print(reply2)

Below are two tiny, self‑contained snippets that illustrate **one list** and **one tuple** in action.  
Each example includes a short comment that explains what’s happening, so you can see the difference right away.

---

## 📋 Example 1 – A **list** (mutable)

```python
# A list that holds the names of the fruits we have in the fridge
fridge = ["apple", "banana", "orange"]      # <-- created with square brackets

print("Initial fridge:", fridge)            # ['apple', 'banana', 'orange']

# 1️⃣ Add a new fruit
fridge.append("kiwi")
print("After appending kiwi:", fridge)      # ['apple', 'banana', 'orange', 'kiwi']

# 2️⃣ Change an existing entry
fridge[1] = "blueberry"                     # replace 'banana' with 'blueberry'
print("After replacement:", fridge)        # ['apple', 'blueberry', 'orange', 'kiwi']

# 3️⃣ Remove a fruit
removed = fridge.pop(0)                     # take out the first item ('apple')
print("Removed:", removed)                  # Removed: apple
print("Final frid

## 14. Practice questions

Try these yourself before looking back at the examples.

1. What is the difference between an LLM and a chat model?
2. Why do we use a prompt template instead of writing a new prompt every time?
3. What is the purpose of the `system`, `human`, and `ai` roles?
4. What does `StrOutputParser` return?
5. Explain the LCEL pipe operator using the `prompt | llm | parser` example.
6. What is a chain?
7. What is the difference between `invoke()`, `batch()`, and `stream()`?
8. What does `RunnableParallel` do?
9. What does `RunnableLambda` do?
10. Why does a chat application need chat history?
11. What is a session ID?
12. Why is in-memory chat history not enough for a production application?

### Small coding challenges

- Build a chain that explains a given Python topic in three bullet points.
- Build a chain that translates a sentence into a language supplied as an input variable.
- Use `RunnableParallel` to produce a definition and two use cases for a technology.
- Use chat history to remember a student's name within one session.
- Use `batch()` to generate explanations for three different AWS services.

## 15. Quick revision sheet

| Topic | Remember it as |
|---|---|
| LLM | The language-generation engine |
| Chat model | The engine that works with conversation messages |
| Prompt template | A reusable instruction form with blanks |
| Messages and roles | Labels showing who is speaking |
| Output parser | Converts model output into a useful format |
| LCEL | Connects steps using `|` |
| Chain | A workflow made from connected steps |
| Runnable | A component with standard execution methods |
| `invoke` | Run once |
| `batch` | Run several inputs |
| `stream` | Receive output in chunks |
| Chat history | Earlier messages supplied as context |
| Session ID | Identifies one conversation |

### The core pattern

`Prompt Template → Chat Model → Output Parser`

```python
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "Python"})
```

**Security reminder:** Never commit API keys to GitHub or share notebooks containing real keys. Use environment variables or a secrets manager. In-memory history is temporary; use persistent storage for real applications.